# Dewarping — look at the results

Run this **after** a training session has finished and saved its version. It is
read-only: it trains nothing and changes nothing.

## Setup

**Settings → Accelerator: GPU T4 x2**, **Internet: off**.

**Add Data →** attach four things:

| | |
|---|---|
| PubLayNet Dewarping Code | needs the version that contains `inference.py` |
| PubLayNet Dewarping Layout Store | |
| PubLayNet Dewarping Dataset | |
| **the training session's saved output** | *Your Datasets* or *Your Work → Output* |

Then run every cell. It takes about two minutes.

## What you get

1. the loss curve — is it still improving, or has it flattened?
2. a contact sheet — input, what the model produced, what a perfect flow would
   have produced, and the truth
3. EPE broken down by distortion type

### How to read the contact sheet

Four columns per row:

| column | what it is |
|---|---|
| **input** | the warped page the model was given |
| **PREDICTED** | the model's dewarp, with ground-truth layout boxes drawn on |
| **rectified with TRUE flow** | the same page dewarped with the *correct* flow — the ceiling this pipeline can reach |
| **ground truth** | the original flat page |

The green boxes are the whole point. In a flat page those boxes *are* axis-aligned
rectangles around text blocks. Drawn on the model's output, they show at a glance
whether dewarped text came out straight: **text that drifts out of its box is
leftover curvature.** Compare column 2 against column 3 — column 3 is what
success looks like, so the gap between them is what is left to learn.

Judge the model by column 2 vs column 3, never column 2 vs column 4. Column 4 is
unreachable: rectification cannot restore detail the warp destroyed.

## 1 — Find everything

In [ ]:
import os, json
from pathlib import Path

def find_dir(marker, depth=7):
    base = Path('/kaggle/input'); hits = []
    for root, dirs, files in os.walk(base):
        if len(Path(root).relative_to(base).parts) > depth:
            dirs[:] = []; continue
        if marker in files:
            hits.append(root)
    return sorted(hits)

CODE   = (find_dir('train.py')    or [''])[0]
LAYOUT = (find_dir('stems.jsonl') or [''])[0]
DATA   = (find_dir('flows.f16')   or [''])[0]

runs = find_dir('last.pt') + [d for d in find_dir('best.pt') if d not in find_dir('last.pt')]
RUN  = runs[0] if runs else ''

for name, p in [('CODE', CODE), ('LAYOUT', LAYOUT), ('DATA', DATA), ('RUN', RUN)]:
    print(f'{name:7s} {p or "** NOT FOUND **"}')
if len(runs) > 1:
    print('\nmore than one run attached:')
    for r in runs: print('   ', r)
    print('set RUN by hand if the first is not the one you want')

assert CODE and LAYOUT and DATA and RUN, 'attach the missing input(s) and rerun'

## 2 — Stage the code

`inference.py` has to be in the Code dataset. If the check below fails, the
notebook is pinned to an older version: remove the Code dataset from the Input
panel, re-add it, and rerun cells 1 and 2.

In [ ]:
import shutil, sys
WORK = Path('/kaggle/working/code'); WORK.mkdir(parents=True, exist_ok=True)
for p in sorted(Path(CODE).glob('*.py')):
    shutil.copy2(p, WORK / p.name)

if not (WORK / 'inference.py').exists():
    raise SystemExit('inference.py is not in this version of the Code dataset.\n'
                     'Re-upload the code, then remove and re-add the input.')
sys.path.insert(0, str(WORK))
print('staged', len(list(WORK.glob('*.py'))), 'modules')

ckpt = Path(RUN) / 'best.pt'
if not ckpt.exists():
    ckpt = Path(RUN) / 'last.pt'
print('checkpoint:', ckpt)

## 3 — The loss curve

`epe_px` is mean end-point error in full-resolution pixels — the headline number,
and the Local Distortion proxy the paper reports. The identity warp ("predict
nothing") scores about **40 px** on the full validation set, so that is the line to be under.

A curve still bending downward at the end of a session means the next session is
worth running. A flat one means the remaining epochs are not buying anything, and
`--epochs` can be cut.

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(Path(RUN) / 'log.jsonl')]

# train.py tags validations four ways: 'val_at_start', 'val' (every 1,000
# steps), 'val_epoch' (every epoch boundary) and 'final'. Train rows carry no
# split at all. Filtering for split == 'val' alone would silently drop the
# epoch-boundary points, which are the ones worth looking at.
VAL = {'val_at_start', 'val', 'val_epoch', 'final'}
tr    = [r for r in rows if r.get('split') is None]
val   = [r for r in rows if r.get('split') in VAL]
epoch = [r for r in rows if r.get('split') == 'val_epoch']
print(f'{len(rows)} rows | {len(tr)} train, {len(val)} validations '
      f'({len(epoch)} at epoch boundaries)')
if val:
    print(f"first: step {val[0]['step']:>8,}  epe {val[0]['epe_px']:7.3f} px")
    print(f"last : step {val[-1]['step']:>8,}  epe {val[-1]['epe_px']:7.3f} px")
    best = min(val, key=lambda r: r['epe_px'])
    print(f"best : step {best['step']:>8,}  epe {best['epe_px']:7.3f} px")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
if tr:
    ax[0].plot([r['step'] for r in tr], [r['epe_px'] for r in tr],
               lw=0.8, alpha=.45, label='train')
if val:
    ax[0].plot([r['step'] for r in val], [r['epe_px'] for r in val],
               'o-', ms=3, lw=1.4, label='val')
if epoch:
    ax[0].plot([r['step'] for r in epoch], [r['epe_px'] for r in epoch],
               's', ms=7, mfc='none', mec='k', label='epoch end')
ax[0].axhline(39.88, ls='--', c='crimson', lw=1, label='identity warp (39.88 px)')
ax[0].set_xlabel('step'); ax[0].set_ylabel('EPE (px)')
ax[0].set_title('end-point error'); ax[0].legend(); ax[0].grid(alpha=.3)

for key, lab in (('flow_l1', 'flow L1'), ('layout', 'layout'),
                 ('photometric', 'photometric')):
    ys = [(r['step'], r[key]) for r in tr if key in r]
    if ys:
        ax[1].plot([y[0] for y in ys], [y[1] for y in ys], lw=0.9, label=lab)
ax[1].set_xlabel('step'); ax[1].set_yscale('log')
ax[1].set_title('loss terms (train)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('/kaggle/working/curve.png', dpi=110); plt.show()

## 4 — The contact sheet

In [ ]:
import torch, cv2
import inference as inf
from dewarp_dataset import DewarpTrainingSet
from IPython.display import Image, display

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net, info = inf.load_checkpoint(ckpt, dev)
print(f"base={info['base']}  epoch {info.get('epoch')}  step {info.get('gstep'):,}")

summary = inf.preview_from_dataset(
    net, DATA, LAYOUT, n=6, device=dev,
    out_path='/kaggle/working/preview.png', split='val', seed=0)

img = cv2.imread('/kaggle/working/preview.png')
cv2.imwrite('/kaggle/working/preview_small.png',
            cv2.resize(img, (img.shape[1] // 2, img.shape[0] // 2),
                       interpolation=cv2.INTER_AREA))
display(Image(filename='/kaggle/working/preview_small.png'))

## 5 — EPE by distortion type

The aggregate hides the thing worth knowing. `combined` is the hardest profile
and the closest to a real photograph; a model can sit at a respectable average
while failing on it completely, and one number would never say so.

`flat` and `cylinder` usually come out about level (flat has only ~34 samples and
still includes camera framing), so do not read a winner between those two. If
`combined` is NOT the worst of the four, something is wrong beyond "needs more training".

In [ ]:
val_ds = DewarpTrainingSet(DATA, split='val', layout_root=LAYOUT, verbose=False)
res, cnt = inf.epe_by_profile(net, val_ds, n=400, device=dev, batch=16)
print(f"{'profile':12s} {'EPE px':>9s} {'n':>6s}")
for k, v in res.items():
    print(f"{k:12s} {v:9.3f} {cnt.get(k, sum(cnt.values())):6d}")

json.dump({'checkpoint': str(ckpt), 'step': info.get('gstep'),
           'epe_by_profile': res, 'preview': summary},
          open('/kaggle/working/eval_summary.json', 'w'), indent=2)
print('\nwrote /kaggle/working/eval_summary.json')

## 6 — Keep it

These files are in `/kaggle/working` and vanish unless you save.

**Save Version → Quick Save**, with output included. Then `preview.png`,
`curve.png` and `eval_summary.json` are downloadable and keep a record of what
the model looked like at this step — which is what you want when comparing the
layout arm against the ablation later.